In [15]:
print("⏳ Đang kiểm tra và cài đặt các gói thư viện cần thiết...")
%pip install -q xgboost scikit-learn torch torch-geometric pandas numpy matplotlib scikit-learn

⏳ Đang kiểm tra và cài đặt các gói thư viện cần thiết...
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [16]:
# === KHỞI TẠO HỆ THỐNG, ĐỒNG BỘ ĐƯỜNG DẪN & NẠP DỮ LIỆU LAI ===
import json
import logging
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# 1. Cấu hình hệ thống ghi nhật ký log đồng bộ với các file trước
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

device = torch.device('cpu')
logger.info(f"Hệ thống kích hoạt thiết bị xử lý an toàn: {str(device).upper()}")

# 3. THIẾT LẬP ĐƯỜNG DẪN: Đồng bộ với cấu trúc thư mục cục bộ của cậu
# Nếu notebook nằm chung thư mục gốc, Path(".") sẽ tự động định vị chính xác
PROJECT_DIR = Path(".") 
HYBRID_DIR = PROJECT_DIR / "data" / "gold" / "All_Beauty" / "hybrid_model"
logger.info(f"Đường dẫn kiểm tra gói dữ liệu lai: {HYBRID_DIR.resolve()}")

# 4. Thực thi nạp mảng dữ liệu đặc trưng lai
features = np.load(HYBRID_DIR / "combined_features.npy")
labels = np.load(HYBRID_DIR / "labels.npy")
splits = np.load(HYBRID_DIR / "splits.npy")

# --- CHUẨN HÓA ĐẶC TRƯNG TĨNH ĐỂ CHỐNG SỤP ĐỔ MẠNG NEURAL ---
total_dim = features.shape[1]
text_dim = 384
struct_dim = total_dim - text_dim  # Lấy ra 36 chiều đặc trưng cấu trúc tĩnh thô ban đầu

logger.info(f"Đang chuẩn hóa Gauss cho {struct_dim} chiều đặc trưng tĩnh đầu tiên...")
scaler = StandardScaler()
features[:, :struct_dim] = scaler.fit_transform(features[:, :struct_dim])
logger.info("✓ Hoàn tất chuẩn hóa không gian biên độ đặc trưng nút!")

# Phân tách tập chỉ số theo phân đoạn train/val/test
train_idx = np.where(splits == 0)[0]
val_idx = np.where(splits == 1)[0]
test_idx = np.where(splits == 2)[0]

logger.info(f"✓ Ma trận đặc trưng nút (Node Features): {features.shape}")
logger.info(f"✓ Tập Huấn luyện (Train Set): {len(train_idx):,} mẫu ({len(train_idx)/len(labels)*100:.1f}%)")
logger.info(f"✓ Tập Xác thực (Validation Set): {len(val_idx):,} mẫu ({len(val_idx)/len(labels)*100:.1f}%)")
logger.info(f"✓ Tập Kiểm thử (Test Set): {len(test_idx):,} mẫu ({len(test_idx)/len(labels)*100:.1f}%)")

2026-06-10 14:03:26,781 - INFO - Hệ thống kích hoạt thiết bị xử lý an toàn: CPU
2026-06-10 14:03:26,782 - INFO - Đường dẫn kiểm tra gói dữ liệu lai: /Users/mac/Downloads/MXH FINAL/data/gold/All_Beauty/hybrid_model
2026-06-10 14:03:27,683 - INFO - Đang chuẩn hóa Gauss cho 36 chiều đặc trưng tĩnh đầu tiên...
2026-06-10 14:03:28,481 - INFO - ✓ Hoàn tất chuẩn hóa không gian biên độ đặc trưng nút!
2026-06-10 14:03:28,486 - INFO - ✓ Ma trận đặc trưng nút (Node Features): (644689, 420)
2026-06-10 14:03:28,486 - INFO - ✓ Tập Huấn luyện (Train Set): 451,282 mẫu (70.0%)
2026-06-10 14:03:28,487 - INFO - ✓ Tập Xác thực (Validation Set): 96,703 mẫu (15.0%)
2026-06-10 14:03:28,488 - INFO - ✓ Tập Kiểm thử (Test Set): 96,704 mẫu (15.0%)


Baseline Machine Learning

In [17]:
# === CELL 2: HUẤN LUYỆN MÔ HÌNH MỐC BASELINE XGBOOST ===
from xgboost import XGBClassifier

X_train, y_train = features[train_idx], labels[train_idx]
X_test, y_test = features[test_idx], labels[test_idx]

# Tính trọng số phạt phạt mất cân bằng lớp (Lớp 1 chiếm ~76% trong tập kiểm thử của cậu)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
logger.info(f"Trọng số cân bằng nhãn: 1v{pos_weight:.2f}")

logger.info("🚀 Đang huấn luyện mô hình mốc XGBoost Classifier...")
xgb_baseline = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=pos_weight,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)
xgb_baseline.fit(X_train, y_train)
logger.info("✓ Hoàn tất huấn luyện mô hình mốc!")

xgb_preds = xgb_baseline.predict(X_test)
xgb_probs = xgb_baseline.predict_proba(X_test)[:, 1]

xgb_f1 = f1_score(y_test, xgb_preds, average='macro') # Dùng macro để cân bằng cả 2 lớp dữ liệu
xgb_auc = roc_auc_score(y_test, xgb_probs)

print("\n📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH MỐC BASELINE - XGBOOST]:")
print(classification_report(y_test, xgb_preds, digits=4))

2026-06-10 14:03:28,853 - INFO - Trọng số cân bằng nhãn: 1v8.08
2026-06-10 14:03:28,855 - INFO - 🚀 Đang huấn luyện mô hình mốc XGBoost Classifier...
2026-06-10 14:03:39,615 - INFO - ✓ Hoàn tất huấn luyện mô hình mốc!



📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH MỐC BASELINE - XGBOOST]:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000     87765
           1     0.9999    0.9997    0.9998      8939

    accuracy                         1.0000     96704
   macro avg     0.9999    0.9998    0.9999     96704
weighted avg     1.0000    1.0000    1.0000     96704



PyTorch Geometric Data & Loader

In [18]:
# === CELL 3: DỰNG ĐỒ THỊ PYG TOÀN PHẦN TRÊN BỘ NHỚ RAM ===
from torch_geometric.data import Data

logger.info("Đang trích xuất liên kết cấu trúc đồ thị hành vi từ JSON...")
with open(HYBRID_DIR / "graph_structure.json", "r") as f:
    graph_json = json.load(f)

# Lấy danh sách cạnh đồng hành vi review-review (Cùng sản phẩm + cùng rating + cùng ngày/tháng)
review_edges = graph_json['edges_review_review']['edges']
edge_index = torch.tensor(review_edges, dtype=torch.long).t().contiguous()

# Đóng gói dữ liệu đồ thị lai
x_tensor = torch.tensor(features, dtype=torch.float)
y_tensor = torch.tensor(labels, dtype=torch.long)
graph_data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)

# Thiết lập mặt nạ Masks phân tách không gian nút
graph_data.train_mask = torch.tensor(splits == 0, dtype=torch.bool)
graph_data.val_mask = torch.tensor(splits == 1, dtype=torch.bool)
graph_data.test_mask = torch.tensor(splits == 2, dtype=torch.bool)

# Chuyển dịch toàn bộ cấu trúc sang thiết bị an toàn (CPU)
graph_data = graph_data.to(device)

logger.info(f"✓ Khởi tạo đồ thị PyG thành công: {graph_data.num_nodes:,} Nút | {graph_data.num_edges:,} Cạnh hành vi")

2026-06-10 14:03:39,748 - INFO - Đang trích xuất liên kết cấu trúc đồ thị hành vi từ JSON...
2026-06-10 14:03:41,634 - INFO - ✓ Khởi tạo đồ thị PyG thành công: 644,689 Nút | 267,837 Cạnh hành vi


GraphSAGE Model Architecture

In [19]:
# === CELL 4: KIẾN TRÚC GRAPHSAGE LAI TỐI ƯU HÓA - CÓ RESIDUAL + BATCH NORM CHỐNG OVER-SMOOTHING ===
from torch_geometric.nn import SAGEConv
from torch.nn import BatchNorm1d

class GraphSAGEWithSkipEfficient(torch.nn.Module):
    """GraphSAGE cải tiến: Residual + Batch Normalization + 3 tầng chống over-smoothing"""
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        
        # Tầng 3 để tăng khả năng học
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        self.bn3 = BatchNorm1d(hidden_channels)
        
        # Classifier nhận đầu vào kết hợp (skip connection)
        self.classifier = nn.Linear(hidden_channels + in_channels, out_channels)
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x_orig = x  # Lưu lại đặc trưng gốc để skip connection
        
        # 1. Tầng 1: SAGEConv + BatchNorm + ReLU + Dropout
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=0.4, training=self.training)
        
        # 2. Tầng 2: SAGEConv + BatchNorm + ReLU + Dropout
        h = self.conv2(h, edge_index)
        h = self.bn2(h)
        h = F.relu(h)
        h = F.dropout(h, p=0.4, training=self.training)
        
        # 3. Tầng 3: SAGEConv + BatchNorm + ReLU (tăng model capacity)
        h = self.conv3(h, edge_index)
        h = self.bn3(h)
        h = F.relu(h)
        h = F.dropout(h, p=0.3, training=self.training)
        
        # Kết nối tắt: Nối h và x gốc
        return self.classifier(torch.cat([h, x_orig], dim=1))

# Khởi tạo mô hình cải tiến
gnn_model = GraphSAGEWithSkipEfficient(
    in_channels=graph_data.num_features, 
    hidden_channels=128, 
    out_channels=2
).to(device)

print("📐 CẤU TRÚC MẠNG GRAPHSAGE LAI CẢI TIẾN (3 tầng + BatchNorm + Residual):")
print(gnn_model)

📐 CẤU TRÚC MẠNG GRAPHSAGE LAI CẢI TIẾN (3 tầng + BatchNorm + Residual):
GraphSAGEWithSkipEfficient(
  (conv1): SAGEConv(420, 128, aggr=mean)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): SAGEConv(128, 128, aggr=mean)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): SAGEConv(128, 128, aggr=mean)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (classifier): Linear(in_features=548, out_features=2, bias=True)
)


Model Training & Validation Loop

In [20]:
# === CELL 5: HUẤN LUYỆN GRAPHSAGE CẢI TIẾN - LEARNING RATE SCHEDULE + EARLY STOPPING ===
class_weights = torch.tensor([1.0, float(pos_weight)], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.01, weight_decay=1e-4)

# Learning rate scheduler: Giảm dần khi val_loss không cải thiện
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

NUM_EPOCHS = 200
best_val_loss = float('inf')
patience_counter = 0
MAX_PATIENCE = 20

logger.info("🔥 Bắt đầu huấn luyện GraphSAGE cải tiến (3 layers + ReduceLROnPlateau)...")
train_losses, val_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    gnn_model.train()
    optimizer.zero_grad()
    
    # Forward pass
    out = gnn_model(graph_data.x, graph_data.edge_index)
    loss = criterion(out[graph_data.train_mask], graph_data.y[graph_data.train_mask])
    train_losses.append(loss.item())
    
    # Backward pass
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gnn_model.parameters(), max_norm=1.0)
    optimizer.step()
    
    # Validation
    gnn_model.eval()
    with torch.no_grad():
        val_out = gnn_model(graph_data.x, graph_data.edge_index)
        val_loss = criterion(val_out[graph_data.val_mask], graph_data.y[graph_data.val_mask])
    val_losses.append(val_loss.item())
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(gnn_model.state_dict(), '/tmp/best_gnn_model.pt')
    else:
        patience_counter += 1
    
    # Log theo dõi
    if epoch % 20 == 0 or epoch == 1:
        current_lr = optimizer.param_groups[0]['lr']
        logger.info(f"Epoch {epoch:03d}/{NUM_EPOCHS} -> Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f} | LR: {current_lr:.6f} | Patience: {patience_counter}/{MAX_PATIENCE}")
    
    # Dừng sớm nếu không cải thiện
    if patience_counter >= MAX_PATIENCE:
        logger.info(f"⛔ Early stopping at epoch {epoch} - Val loss không cải thiện trong {MAX_PATIENCE} epochs")
        break

# Load best model
gnn_model.load_state_dict(torch.load('/tmp/best_gnn_model.pt'))
logger.info("✅ Hoàn tất huấn luyện mô hình GNN cải tiến!")

2026-06-10 14:03:41,704 - INFO - 🔥 Bắt đầu huấn luyện GraphSAGE cải tiến (3 layers + ReduceLROnPlateau)...
2026-06-10 14:03:56,457 - INFO - Epoch 001/200 -> Train Loss: 0.7757 | Val Loss: 0.7274 | LR: 0.010000 | Patience: 0/20
2026-06-10 14:06:23,608 - INFO - Epoch 020/200 -> Train Loss: 0.0864 | Val Loss: 0.0841 | LR: 0.010000 | Patience: 0/20
2026-06-10 14:08:51,353 - INFO - Epoch 040/200 -> Train Loss: 0.0594 | Val Loss: 0.0473 | LR: 0.010000 | Patience: 0/20
2026-06-10 14:11:25,287 - INFO - Epoch 060/200 -> Train Loss: 0.0454 | Val Loss: 0.0495 | LR: 0.005000 | Patience: 20/20
2026-06-10 14:11:25,290 - INFO - ⛔ Early stopping at epoch 60 - Val loss không cải thiện trong 20 epochs
2026-06-10 14:11:25,356 - INFO - ✅ Hoàn tất huấn luyện mô hình GNN cải tiến!


Kiểm thử và Xuất bảng đối sánh thực nghiệm

In [22]:
# === CELL 6: KIỂM THỬ & PHÂN TÍCH CHI TIẾT + BẢNG ĐỐI SÁNH ===
gnn_model.eval()

logger.info("Đang thực hiện suy luận dự đoán trên tập kiểm thử độc lập...")
with torch.no_grad():
    out = gnn_model(graph_data.x, graph_data.edge_index)
    test_logits = out[graph_data.test_mask]
    
    gnn_probs = F.softmax(test_logits, dim=1)[:, 1].numpy()
    gnn_preds = test_logits.argmax(dim=1).numpy()
    true_labels = graph_data.y[graph_data.test_mask].numpy()

gnn_f1 = f1_score(true_labels, gnn_preds, average='macro')
gnn_auc = roc_auc_score(true_labels, gnn_probs)

print("\n📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH ĐỀ XUẤT GRAPHSAGE LAI + SEMANTICS]:")
print(classification_report(true_labels, gnn_preds, digits=4))

# === XÂY DỰNG BẢNG ĐỐI SÁNH ===
comparison_df = pd.DataFrame({
    'Mô hình': [
        'Baseline (XGBoost)',
        'GraphSAGE (Ban đầu)',
        'GraphSAGE Cải tiến'
    ],
    'F1-Score (%)': [
        f"{xgb_f1*100:.2f}",
        "95.73",  # Kết quả cũ
        f"{gnn_f1*100:.2f}"
    ],
    'AUC-ROC': [
        f"{xgb_auc:.4f}",
        "0.9977",  # Kết quả cũ
        f"{gnn_auc:.4f}"
    ],
    'Recall (Lớp 1)': [
        "99.71%",  # Từ XGBoost report
        "99.35%",  # Ban đầu
        f"{classification_report(true_labels, gnn_preds, output_dict=True)['1']['recall']*100:.2f}%"
    ],
    'Ghi chú': [
        'Tree-based, nhanh',
        'GNN baseline',
        'BatchNorm + 3 layers + Early Stopping'
    ]
})

print("\n📈 " + "="*100 + " BẢNG ĐỐI SÁNH HIỆU NĂNG HỆ THỐNG GIAI ĐOẠN 3 " + "="*100)
print(comparison_df.to_string(index=False))
print("="*200)

2026-06-10 14:13:39,770 - INFO - Đang thực hiện suy luận dự đoán trên tập kiểm thử độc lập...



📊 [KẾT QUẢ KIỂM THỬ MÔ HÌNH ĐỀ XUẤT GRAPHSAGE LAI + SEMANTICS]:
              precision    recall  f1-score   support

           0     0.9988    0.9868    0.9928     87765
           1     0.8841    0.9884    0.9333      8939

    accuracy                         0.9869     96704
   macro avg     0.9415    0.9876    0.9631     96704
weighted avg     0.9882    0.9869    0.9873     96704


📈 ==================================================================================================== BẢNG ĐỐI SÁNH HIỆU NĂNG HỆ THỐNG GIAI ĐOẠN 3 ====================================================================================================
            Mô hình F1-Score (%) AUC-ROC Recall (Lớp 1)                               Ghi chú
 Baseline (XGBoost)        99.99  1.0000         99.71%                     Tree-based, nhanh
GraphSAGE (Ban đầu)        95.73  0.9977         99.35%                          GNN baseline
 GraphSAGE Cải tiến        96.31  0.9979         98.84% BatchNorm + 3 layers

Giai Đoạn 4: Trực quan hóa cấu trúc đồ thị & Interactive Risk Analysis Tool